# Labeled evaluation: metrics, calibration, statistical tests, bias/variance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/evaluation.ipynb)

Companion notebook for the ReliableML paper `docs/latex/evaluation.tex`. It calls `ReliablePipeline.evaluate`, which adds label-dependent analyses to an ordinary prediction: confusion-matrix metrics (including MCC), probability calibration, ANOVA/MANOVA/chi-square tests and a repeated-CV Brier bias/variance decomposition.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Metrics

With confusion matrix $C\in\mathbb N^{K\times K}$ ($C_{ij}$: true class $i$, predicted $j$), $s=\sum_{ij}C_{ij}$, $c=\sum_k C_{kk}$, $t_k=\sum_j C_{kj}$, $p_k=\sum_i C_{ik}$:
$$\operatorname{acc}=\frac{c}{s},\qquad \operatorname{BA}=\frac1K\sum_k\frac{C_{kk}}{t_k},\qquad
\operatorname{MCC}=\frac{c\,s-\sum_k p_kt_k}{\sqrt{(s^2-\sum_kp_k^2)(s^2-\sum_kt_k^2)}}.$$
Per class (one-vs-rest): precision, recall (TPR), specificity (TNR) and $F_1$, plus macro and support-weighted averages. MCC is $0$ for chance-level and can be low when accuracy is high under heavy imbalance.

**Calibration** (needs `predict_proba`): the multiclass Brier score $\frac1n\sum_i\sum_k(\hat p_{ik}-y_{ik})^2$ (class-wise mean reported), log loss, and the top-label expected calibration error over confidence bins $B_m$:
$$\operatorname{ECE}=\sum_m\frac{\lvert B_m\rvert}{n}\,\bigl\lvert\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\bigr\rvert.$$

**Association tests** (descriptive, assumption-dependent): per numeric feature one-way ANOVA across labels, $F=\frac{\mathrm{SS}_B/(K-1)}{\mathrm{SS}_W/(n-K)}$; a joint MANOVA with Wilks' $\Lambda=\det(E)/\det(E+H)$; chi-square for categorical features. The same tests compare training and test cohorts.

**Bias/variance.** Repeated stratified CV gives out-of-fold probability vectors $\hat p^{(r)}_i$ over repeats $r$. With the one-hot label $y_i$ and mean prediction $\bar p_i$:
$$\underbrace{\overline{\lVert\bar p_i-y_i\rVert^2}}_{\text{bias}^2}+\underbrace{\overline{\tfrac1R\textstyle\sum_r\lVert\hat p^{(r)}_i-\bar p_i\rVert^2}}_{\text{variance}}=\text{mean squared probability error}.$$
This is an *empirical* decomposition of Brier error, not a population irreducible-noise estimate.

## 2. Binary example: breast cancer

In [ ]:
import json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
from ml_reliability import ReliablePipeline

warnings.filterwarnings("ignore", category=UserWarning)
data = load_breast_cancer(as_frame=True)
X = data.data[["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean symmetry"]]
y = data.target  # 0 = malignant, 1 = benign
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)

pipe = ReliablePipeline(
    RandomForestClassifier(n_estimators=100, random_state=0),
    plot_dir="reliability_plots", bias_variance_folds=3, bias_variance_repeats=3, random_seed=0,
).fit(Xtr, ytr)
report = pipe.evaluate(Xte, yte, positive_label=1)
result = report.to_dict()
cls = result["classification"]
print("n =", result["sample_count"], "| labels:", cls["labels"])
print({k: round(cls[k], 4) for k in ["accuracy", "balanced_accuracy", "matthews_correlation_coefficient"]})
print(pd.DataFrame(cls["per_class"]).round(3).to_string(index=False))

In [ ]:
cm = np.array(cls["confusion_matrix"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(len(cm)), cls["labels"]); axes[0].set_yticks(range(len(cm)), cls["labels"])
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true"); axes[0].set_title("Confusion matrix")
for i in range(len(cm)):
    for j in range(len(cm)):
        axes[0].text(j, i, cm[i, j], ha="center", va="center")

cal = result["classification"]["probability_calibration"]
bins = pd.DataFrame(cal["confidence_bins"])
bins = bins[bins["sample_count"] > 0]
axes[1].plot([0, 1], [0, 1], "k--", label="perfect")
axes[1].scatter(bins["mean_confidence"], bins["accuracy"], s=20 + 4 * bins["sample_count"], label="bins (size ~ count)")
axes[1].set_xlabel("mean confidence"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Reliability diagram")
axes[1].legend()
plt.tight_layout(); plt.show()
print({k: round(v, 4) for k, v in cal.items() if isinstance(v, float)})

### Verify the headline metrics against sklearn and the formulas

In [ ]:
pred = pipe.predict(Xte)
print("MCC report :", round(cls["matthews_correlation_coefficient"], 6))
print("MCC sklearn:", round(matthews_corrcoef(yte, pred), 6))
s = cm.sum(); c = np.trace(cm); t = cm.sum(axis=1); p = cm.sum(axis=0)
print("MCC formula:", round((c * s - (p * t).sum()) / np.sqrt((s**2 - (p**2).sum()) * (s**2 - (t**2).sum())), 6))
print("accuracy    :", round(cls["accuracy"], 6), "vs", round(c / s, 6))

## 3. Association tests and bias/variance

In [ ]:
tests = result["statistical_tests"]
anova = pd.DataFrame(tests["feature_target_associations"]["anova_numeric_features"])
print(anova[["feature", "statistic", "p_value", "group_sizes"]].round(4).to_string(index=False))
print("MANOVA (features vs label):")
print(json.dumps(tests["feature_target_associations"]["manova_numeric_features"], indent=1, default=str)[:900])
print("\nTrain-vs-test cohort ANOVA (a split-balance check):")
cohort = pd.DataFrame(tests["training_vs_test_cohorts"]["anova_numeric_features"])
print(cohort[["feature", "statistic", "p_value"]].round(4).to_string(index=False))
bv = result["bias_variance"]
print("\nBias/variance:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in bv.items() if k in
      ["bias_squared", "prediction_variance", "mean_squared_probability_error", "folds", "repeats"]})

## 4. Multiclass example: wine

Multiclass tasks do not take `positive_label`; per-class metrics are one-vs-rest.

In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data[["alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium", "flavanoids"]]
Xwtr, Xwte, ywtr, ywte = train_test_split(Xw, wine.target, test_size=0.4, random_state=0, stratify=wine.target)
wine_pipe = ReliablePipeline(LogisticRegression(max_iter=2000), plot_dir="reliability_plots",
                             bias_variance_folds=3, bias_variance_repeats=3, random_seed=0).fit(Xwtr, ywtr)
wine_res = wine_pipe.evaluate(Xwte, ywte).to_dict()["classification"]
print("labels:", wine_res["labels"])
print(np.array(wine_res["confusion_matrix"]))
print({k: round(wine_res[k], 4) for k in ["accuracy", "balanced_accuracy", "matthews_correlation_coefficient"]})
print("MCC sklearn:", round(matthews_corrcoef(ywte, wine_pipe.predict(Xwte)), 4))
print(pd.DataFrame(wine_res["per_class"]).round(3).to_string(index=False))

## 5. Why accuracy alone misleads: synthetic imbalanced data

In [ ]:
rng = np.random.default_rng(1)
n = 3000
Xs = pd.DataFrame(rng.normal(size=(n, 4)), columns=[f"x{i}" for i in range(4)])
logit = -4.0 + 1.2 * Xs["x0"] + 0.8 * Xs["x1"]  # about 3% positives, weak signal
ys = pd.Series((rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int))
print("positive rate:", round(ys.mean(), 4))
Xstr, Xste, ystr, yste = train_test_split(Xs, ys, test_size=0.4, random_state=0, stratify=ys)
imb = ReliablePipeline(LogisticRegression(max_iter=1000), plot_dir="reliability_plots",
                       bias_variance_folds=3, bias_variance_repeats=3, random_seed=0).fit(Xstr, ystr)
imb_cls = imb.evaluate(Xste, yste, positive_label=1).to_dict()["classification"]
print({k: round(imb_cls[k], 4) for k in ["accuracy", "balanced_accuracy", "matthews_correlation_coefficient"]})
print("confusion matrix:\n", np.array(imb_cls["confusion_matrix"]))
print("majority-class baseline accuracy:", round(1 - yste.mean(), 4))

## Interpretation and limitations

**How to read the output.** Accuracy, balanced accuracy and MCC answer different questions; the imbalanced example shows accuracy close to the majority-class baseline while recall for the rare class can be poor. The reliability diagram compares confidence with observed accuracy bin by bin (sparse bins are noisy). Treat ANOVA/MANOVA p-values as screening summaries. The bias/variance numbers are an empirical Brier decomposition from out-of-fold predictions on the evaluated batch, so with a single repeat the variance term is zero by construction.

**Limitations.** Metrics on a small test set carry substantial sampling error (no intervals are shown for the point metrics here); ANOVA/MANOVA assume independence, normality and equal variances, and are not tested; multiple tests are not corrected; ECE depends on binning; cross-validation refits clones of the estimator on the evaluated batch, so the decomposition describes the learning procedure, not the fitted model; none of it is causal.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.